# Diabetes Prediction with PyTorch + MLflow

## Step 1: Installing Dependencies

In [1]:
!pip install mlflow torch scikit-learn pandas numpy requests --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Importing Libraries

In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.pyfunc
import time

## Step 3: Loading Data

In [3]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, header=None, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Step 4: Exploring Data

In [4]:
print(df.shape)
print(df.describe())
print(df['Outcome'].value_counts())

(768, 9)
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.000000         

## Step 5: Preprocessing & Splitting

In [5]:
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

X_train, X_rem, y_train, y_rem = train_test_split(X, y, train_size=0.6, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

## Step 6: PyTorch Dataset & DataLoader

In [6]:
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(DiabetesDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(DiabetesDataset(X_val,   y_val),   batch_size=32)

## Step 7: Defining MLP Model

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

## Step 8: Training with MLflow Tracking

In [9]:
def train_model(hidden_dim=32, dropout=0.3, lr=1e-3, epochs=30, run_name='mlp_run'):
    model = MLP(hidden_dim=hidden_dim, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({'hidden_dim': hidden_dim, 'dropout': dropout, 'lr': lr, 'epochs': epochs})

        for epoch in range(1, epochs + 1):
            # Training
            model.train()
            train_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # Validation
            model.eval()
            val_preds, val_labels = [], []
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    val_preds.extend(model(X_batch).numpy())
                    val_labels.extend(y_batch.numpy())

            val_auc = roc_auc_score(val_labels, val_preds)
            avg_loss = train_loss / len(train_loader)
            mlflow.log_metrics({'train_loss': avg_loss, 'val_auc': val_auc}, step=epoch)

            if epoch % 5 == 0:
                print(f'Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Val AUC: {val_auc:.4f}')

        # Log model
        mlflow.pytorch.log_model(model, name='mlp_model')
        run_id = mlflow.active_run().info.run_id

    return run_id, model

run_id, trained_model = train_model(run_name='baseline_mlp')

Epoch 05 | Loss: 0.5724 | Val AUC: 0.8185
Epoch 10 | Loss: 0.5078 | Val AUC: 0.8306
Epoch 15 | Loss: 0.4669 | Val AUC: 0.8379
Epoch 20 | Loss: 0.4772 | Val AUC: 0.8363
Epoch 25 | Loss: 0.4470 | Val AUC: 0.8329


2026/03/13 22:49:47 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 30 | Loss: 0.4613 | Val AUC: 0.8279


## Step 9: Running a Second Experiment for comparison in MLflow UI)

In [10]:
# Try a larger model with less dropout — compare both runs in the MLflow UI
run_id_2, _ = train_model(hidden_dim=64, dropout=0.1, lr=5e-4, run_name='larger_mlp')

Epoch 05 | Loss: 0.5701 | Val AUC: 0.7981
Epoch 10 | Loss: 0.4824 | Val AUC: 0.8296
Epoch 15 | Loss: 0.4629 | Val AUC: 0.8375
Epoch 20 | Loss: 0.4425 | Val AUC: 0.8373
Epoch 25 | Loss: 0.4396 | Val AUC: 0.8338


2026/03/13 22:49:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 30 | Loss: 0.4361 | Val AUC: 0.8329


## Step 10: Registering Model in MLflow Model Registry

In [11]:
model_name = 'diabetes_mlp'
model_version = mlflow.register_model(f'runs:/{run_id}/mlp_model', model_name)
time.sleep(10)
print(f'Registered: {model_name} v{model_version.version}')

Successfully registered model 'diabetes_mlp'.
2026/03/13 22:49:58 WARNING mlflow.tracking._model_registry.fluent: Run with id 9892890e857d4758a05a380989f784cf has no artifacts at artifact path 'mlp_model', registering model based on models:/m-f0779684c3544209b21d6f93602fcb56 instead
Created version '1' of model 'diabetes_mlp'.


Registered: diabetes_mlp v1


## Step 11: Transition Model to Production

In [12]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.transition_model_version_stage(
    name=model_name,
    version=model_version.version,
    stage='Production'
)
print(f'Model transitioned to Production')

Model transitioned to Production


/tmp/ipykernel_298316/938076100.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## Step 12: Loading Production Model & Evaluating on Test Set

In [13]:
prod_model = mlflow.pytorch.load_model(f'models:/{model_name}/production')
prod_model.eval()

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    test_preds = prod_model(X_test_tensor).numpy()

test_auc = roc_auc_score(y_test, test_preds)
print(f'Test AUC: {test_auc:.4f}')

Test AUC: 0.8201


## Step 13: Serve the Model (Terminal Command)

Run in terminal:
```bash
mlflow models serve -m models:/diabetes_mlp/production -h 0.0.0.0 -p 5001 --no-conda
```

## Step 14: Real-Time Inference via REST API

In [14]:
import requests

feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

sample = X_test[:3].tolist()

url = 'http://localhost:5001/invocations'
payload = {
    "dataframe_split": {
        "columns": feature_names,
        "data": sample
    }
}

response = requests.post(url, json=payload)
print('Predictions:', response.json())

Predictions: {'predictions': [{'0': 0.015670524910092354}, {'0': 0.6837818026542664}, {'0': 0.0008723650826141238}]}


## Step 15: Launch MLflow UI (Terminal Command)

Run in terminal to browse all runs, compare metrics, and inspect artifacts:
```bash
mlflow ui
```
Open: http://localhost:5000